In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import tiktoken

red  = "#E61E14"
palette_ek = ['#67001F', '#E61E14', '#D6604D', '#F4A582', '#FFCC66']

sns.set_palette(sns.color_palette(palette_ek))

# working with base environment

# set pandas options
pd.set_option('display.max_columns', None)

In [ ]:
# get working directory
import os
os.getcwd()

# set working directory
os.chdir('analyses')

## NOS

In [ ]:
# Read nos data

nos_articles = pd.read_csv('data/cleaned/to_analyze/final_nosarticles.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(len(nos_articles))

# change page_id to int
nos_articles['page_id'] = nos_articles['page_id'].astype(int)
nos_articles.head()

In [ ]:
# check data types
nos_articles.dtypes

# change Date to date type
nos_articles['Date'] = pd.to_datetime(nos_articles['Date'], format='%Y-%m-%d')

In [ ]:
nos_articles.head()
nr_articles_perdate = nos_articles.groupby('Date')['page_id'].nunique().reset_index(name='nr_articles')
nr_articles_perdate.nr_articles.describe()

# Plotting the data, three subplots per year
plt.subplots(figsize=(30, 20))

# Plot 1
ax = plt.subplot(3, 1, 1)
data1 = nr_articles_perdate[nr_articles_perdate['Date'] <= "2020-12-31"]
ax.plot(data1['Date'], data1['nr_articles'], marker='o', color=red, linewidth=1)  # Change line color to red
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_ylabel('Aantal artikelen', fontsize=14)

# Plot 2
ax2 = plt.subplot(3, 1, 2)
data2 = nr_articles_perdate[(nr_articles_perdate['Date'] <= "2021-12-31") & (nr_articles_perdate['Date'] >= "2021-01-01")]
ax2.plot(data2['Date'], data2['nr_articles'], marker='o', color=red, linewidth=1)  # Change line color to red
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
ax2.grid(True, which='both', linestyle='--', linewidth=0.5)
ax2.set_ylabel('Aantal artikelen', fontsize=14)

# Plot 3
ax3 = plt.subplot(3, 1, 3)
data3 = nr_articles_perdate[(nr_articles_perdate['Date'] <= "2022-12-31") & (nr_articles_perdate['Date'] >= "2022-01-01")]
ax3.plot(data3['Date'], data3['nr_articles'], marker='o', color=red, linewidth=1)  # Change line color to red
ax3.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
ax3.grid(True, which='both', linestyle='--', linewidth=0.5)
ax3.set_ylabel('Aantal artikelen', fontsize=14)


In [ ]:
# # Date should be between December 2019 and May 2022
# nos_articles = nos_articles[(nos_articles['Date'] >= "2019-12-01") & (nos_articles['Date'] <= "2022-05-31")]

In [ ]:
# Set figure size
plt.figure(figsize=(30, 10))

# Plot the data
plt.plot(nr_articles_perdate['Date'], nr_articles_perdate['nr_articles'], marker='o', color='red', linewidth=1)

# Set x-axis and y-axis labels and properties
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
plt.xlabel('Date', fontsize=14)  # Set x-axis label
plt.ylabel('Aantal artikelen', fontsize=14)

# Show grid
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# Display the plot
plt.show()

In [ ]:
# how many weeks are there
# get week number
nos_articles['weekyear'] = nos_articles['Date'].dt.strftime('%Y-%U')
nos_articles['weekyear'].value_counts

nos_articles.head()

In [ ]:
print(len(nos_articles))
print(len(nos_articles['Date'].unique()))

In [ ]:
# calculate number of tokens per article
encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
def num_tokens_from_string(string: str, model_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(model_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

nos_articles["num_tokens"] = nos_articles["Text"].apply(num_tokens_from_string, model_name = "gpt-3.5-turbo")

In [ ]:
nos_articles["num_tokens"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.985, 0.99])

In [ ]:
print(len(nos_articles[nos_articles["num_tokens"] >= 2000].num_tokens.unique()))    
nos_articles[nos_articles["num_tokens"] >= 2000].num_tokens.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
nos_articles[nos_articles["num_tokens"] >= 2000].Owner.value_counts()

In [ ]:
nos_articles[nos_articles["num_tokens"] >= 2000].Keywords.value_counts()

In [ ]:
# get unique categories by splitting the categories column by comma
nos_categories = nos_articles['Category'].str.split(',')

# remove trailing and leading spaces
nos_categories = nos_categories.apply(lambda x: [i.strip() for i in x])
nos_categories

# get unique categories
nos_unique_categories = []
for i in nos_categories:
    for j in i:
        if j not in nos_unique_categories:
            nos_unique_categories.append(j)

print(nos_unique_categories)
print(len(nos_unique_categories))

In [ ]:
# Get the corona keywords
keywords_df = pd.read_csv('data/cleaned/to_analyze/ALLcorona_keywords_list_final_v2.csv', quoting=csv.QUOTE_NONNUMERIC, encoding="utf-8", sep=";")
keywords = keywords_df['Keyword'].tolist()
keywords[0:10]

In [ ]:
def extract_keywords(text):
    text = text.lower()
    matches = re.findall(r'\b(?:' + '|'.join(keywords) + r')\b', text)
    unique_matches = list(set(matches))  # Deduplicate matches
    return ', '.join(unique_matches) if unique_matches else None

In [ ]:
nos_articles['Keywords'] = nos_articles['Keywords'].astype(str)
nos_articles['keyword_check'] = nos_articles['Keywords'].apply(extract_keywords)
nos_articles['keyword_wordcount'] = nos_articles['keyword_check'].str.split().str.len()
nos_articles.head()

In [ ]:
# nos_articles[(nos_articles['keyword_wordcount'] >=1) & (nos_articles['num_tokens'] >= 2000)].num_tokens.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
nos_articles[nos_articles['num_tokens'] >= 2000].keyword_wordcount.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
nos_articles[nos_articles['num_tokens'] >= 2000].keyword_wordcount.value_counts()

nos_articles[(nos_articles['keyword_wordcount'] > 1) & (nos_articles['num_tokens'] >= 2000)].num_tokens.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
nos_articles[(nos_articles['keyword_wordcount'] > 1) & (nos_articles['num_tokens'] >= 2000)].Owner.value_counts()

In [ ]:
nos_articles["num_tokens"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.985, 0.99])

In [ ]:
# limit number of words to 2000 or less
nos_articles = nos_articles[(nos_articles['num_tokens'] <= 2000)]
print(len(nos_articles))

In [ ]:
nos_articles.Title.value_counts()

In [ ]:
# does title include Wekdienst? 
nos_articles['wekdienst'] = nos_articles['Title'].str.contains('Wekdienst')
nos_articles['wekdienst'].value_counts()

# remove if wekdienst is in title
nos_articles = nos_articles[nos_articles['wekdienst'] == False]
len(nos_articles)

In [ ]:
# does title include "uitzending", fist lower the title for the check
nos_articles['Title_lower'] = nos_articles['Title'].str.lower()
nos_articles['uitzending'] = nos_articles['Title_lower'].str.contains('uitzending van')
nos_articles['uitzending'].value_counts()

In [ ]:
# see the Titles when uitzending is True
nos_articles[nos_articles['uitzending'] == True].Title.unique()

# remove if uitzending van is in title
nos_articles = nos_articles[nos_articles['uitzending'] == False]
len(nos_articles)

In [ ]:
nos_articles.head()

In [ ]:
# make variable page_type based on words that come after the 3rd slash in the url
nos_articles['page_type'] = nos_articles['URL'].str.split('/').str[3]
nos_articles['page_type'].value_counts()

In [ ]:
# page_type 2 for the 4th slash
nos_articles['page_type2'] = nos_articles['URL'].str.split('/').str[4]
nos_articles['page_type2'].value_counts()

# final page_type is if page_type2 is artikel then artikel else page_type
nos_articles['page_type_final'] = np.where(nos_articles['page_type2'] == 'artikel', 'artikel', nos_articles['page_type'])
nos_articles['page_type_final'].value_counts()

In [ ]:
nos_articles.Owner.value_counts()

In [ ]:
# see num_words if page_type_final is collectie
nos_articles[nos_articles['page_type_final'] == 'collectie'].num_tokens.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
nos_articles[nos_articles['page_type_final'] == 'collectie'].URL.value_counts()  # all articles

In [ ]:
nos_articles[nos_articles['page_type_final'] == 'nieuwsuur'].URL.value_counts() # all articles

In [ ]:
nos_articles[(nos_articles['num_tokens'] <= 150)].URL.unique()

# remove if num_words is less than 150
nos_articles = nos_articles[(nos_articles['num_tokens'] >= 150)]
len(nos_articles)

In [ ]:
# Decide on the total number of articles to sample
n_samples = 1000
# Number of unique dates
print(nos_articles['weekyear'].nunique())

# Number of unique weeks
n_unique_weeks = nos_articles['weekyear'].nunique()

# Ensure n_samples is at least equal to n_unique_dates
if n_samples < n_unique_weeks:
    print(f"Warning: n_samples should be at least {n_unique_weeks} to ensure a minimum weight of 1 for each date.")
    n_samples = n_unique_weeks

# Compute the number of remaining articles to sample after allocating 1 article per week
n_remaining = n_samples - n_unique_weeks

# Compute the proportion of articles per date
weights = nos_articles['weekyear'].value_counts(normalize=True)

# Allocate the remaining articles based on the computed weights
sampled_counts = (weights * n_remaining).round().astype(int) + 1
len(sampled_counts)

# # Sample based on the computed counts
sampled_articles = pd.concat([nos_articles[nos_articles['weekyear'] == weekyear].sample(n=sample_count) for weekyear, sample_count in sampled_counts.items()])

len(sampled_articles)


In [ ]:
sampled_articles.num_tokens.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
# Set figure size
plt.figure(figsize=(30, 10))

# Plot the data
plt.plot(nr_articles_perdate['Date'], nr_articles_perdate['nr_articles'], marker='o', color='red', linewidth=1)

# Set x-axis and y-axis labels and properties
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
plt.xlabel('Date', fontsize=14)  # Set x-axis label
plt.ylabel('Aantal artikelen', fontsize=14)

# Show grid
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# Display the plot
plt.show()

In [ ]:
nr_articles_perdate = sampled_articles.groupby('Date')['page_id'].nunique().reset_index(name='nr_articles')
nr_articles_perdate.nr_articles.describe()

# Set figure size
plt.figure(figsize=(30, 10))

# Plot the data
plt.plot(nr_articles_perdate['Date'], nr_articles_perdate['nr_articles'], marker='o', color='red', linewidth=1)

# Set x-axis and y-axis labels and properties
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
plt.xlabel('Date', fontsize=14)  # Set x-axis label
plt.ylabel('Aantal artikelen', fontsize=14)

# Show grid
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# Display the plot
plt.show()

In [ ]:
sampled_articles.to_csv('annotation/sampled_articles/sample_nosarticles1000.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
# get the already coded articles
df_coded = pd.read_spss('analyses/annotation/nos_coded/nos_coded_v4.sav')
df_coded.head()

In [ ]:
# change M2 to integer
df_coded['M2'] = df_coded['M2'].astype(int)

In [ ]:
# rename columns M1, M2, M3, M4 
df_coded = df_coded.rename(columns={'M1': 'coder', 
                        'M2': 'article_id', 
                        'M3': 'title', 
                        'M4': 'owner'})

df_coded.head()

In [ ]:
# rename columns M1, M2, M3, M4 
df_coded = df_coded.rename(columns={'V0_': 'about_covid', 
                        'V1._1': 'topic_a', 
                        'V1._2': 'topic_b', 
                        'V1._21': 'topic_c',    
                        'V1._3': 'topic_d',
                        'V1._4': 'topic_e',
                        'V1._5': 'topic_f',
                        'V1._6': 'topic_g',
                        'V1._7': 'topic_h',
                        'V1._8': 'topic_i',
                        'V1._9': 'topic_j',
                        'V1._10': 'topic_k',
                        'V1._11': 'topic_l',
                        'V1._12': 'topic_m',
                        'V1._13': 'topic_n',
                        'V1._15': 'topic_o',
                        'V1._15_TEXT': 'topic_o_text', 
                        'V1.2._2': 'other_country_binary',
                        'V2': 'actors_present'
                        })

In [ ]:
# count the value counts of each column if the column name has the word topic_ in it
topic_columns = [col for col in df_coded.columns if 'topic_' in col]
topic_columns

# get the value counts of each column
for col in topic_columns:
    print(df_coded[col].value_counts())
    print('')

In [ ]:
# check dtypes 
print(df_coded.article_id.dtypes)
print(nos_articles.page_id.dtypes)

In [ ]:
# remove from nos if page_id is in df_coded.article_id
nos_notcoded = nos_articles[~nos_articles.page_id.isin(df_coded.article_id)]
print(len(nos_articles))
print(len(nos_notcoded))
print(len(df_coded))

In [ ]:
nos_notcoded.keyword_wordcount.describe()

In [ ]:
# see the keywords where keyword_wordcount is bigger than 1
nos_notcoded[nos_notcoded.keyword_wordcount > 1].Keywords.value_counts()

In [ ]:
# get categories
nos_notcoded['Category'].str.split(',')

In [ ]:
nos_notcoded[nos_notcoded.keyword_wordcount > 1].Category.value_counts()

In [ ]:
# sample articles where keyword_wordcount is bigger than 1 and Category has the word "Politiek" 
nos_notcoded[(nos_notcoded.keyword_wordcount > 1) & (nos_notcoded.Category.str.contains("Politiek"))].sample(10)

In [ ]:
# get unique words in keywords where keyword_wordcount is bigger than 1 
keywords = nos_notcoded[nos_notcoded.keyword_wordcount > 1].Keywords.str.split().explode().unique()

# save this to an excel
keywords_df = pd.DataFrame(keywords, columns=['Keyword'])

# sort keywords
keywords_df = keywords_df.sort_values(by='Keyword')

keywords_df.to_excel('annotation/annotation_sample_docs/sampled_articles/nos_keywords_notcoded.xlsx', index=False)

In [ ]:
print(len(nos_notcoded))
nos_notcoded = nos_notcoded[nos_notcoded.keyword_wordcount > 1]
print(len(nos_notcoded))


In [ ]:
# see articles where keywords have the word therapie
nos_notcoded[nos_notcoded.Keywords.str.contains('langdurig')]

In [ ]:
len(nos_notcoded[(nos_notcoded.Keywords.str.contains('long covid')) | (nos_notcoded.Description.str.contains('long covid'))| (nos_notcoded.Title.str.contains('long covid'))])

In [ ]:
# check owner in nos_notcoded
nos_notcoded.Owner.value_counts()

print(len(nos_notcoded))

# get only articles where Owner is NOS Nieuws
nos_notcoded_nosnieuws = nos_notcoded[nos_notcoded.Owner == 'NOS Nieuws']

print(len(nos_notcoded_nosnieuws))

In [ ]:
keywords = ['long covid', 'longcovid', 'long-covid', 'langdurige covid', 'lang covid', 'lang-covid', 'post-covid', 'postcovidsyndroom']

long_covid_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Description.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Title.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Text.str.contains('|'.join(keywords), case=False)
]

print(len(long_covid_articles))

long_covid_articles.head()

In [ ]:
keywords = ['desinformatie', 'misinformatie', 'nepnieuws', 'misinformation', 'fake news', 'disinformation', 'fake-news', 'trollen', 'complot', 
            'complottheorie', 'complottheorieën', 'complotdenk', 'disinformatie', 'Viruswaarheid', 'Willem Engel']

misinformation_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Description.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Title.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Text.str.contains('|'.join(keywords), case=False)
]

print(len(misinformation_articles))

misinformation_articles.head()

In [ ]:
keywords = ['gezondheidsraad', 'gezondheidszorg', 'IC-bedden', 'intensive care', 'IC-capaciteit', 'IC-verpleegkundigen', 
            'zorgpersoneel',  'zorgmedewerkers', 'ziekenhuiscapaciteit', 'zorgdruk', 'beddentekort', 'intensivecare']

medische_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False)]
print(len(medische_articles))

medische_articles.head()

In [ ]:
keywords = ['wetenschappers', 'wetenschappelijk', 'viroloog', 'besmettelijkheid', 'epidemologie', 'herbesmettingen', 'herinfecties', 'variant', 'research',
            'behandelmethode', 'bevolkingsonderzoek', 'microbiologen']

wetenschap_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Category.str.contains('|'.join(keywords), case=False)]
print(len(wetenschap_articles))

wetenschap_articles.head()

In [ ]:
keywords = ['financieel', 'economie', 'economisch', 'financiële', 'financiën', 'herstel', 'herstelpakket', 'stimuleringsmaatregelen',
            'steunmaatregelen', 'steunpakket']

economie_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Title.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Description.str.contains('|'.join(keywords), case=False)]
print(len(economie_articles))

economie_articles.head()

In [ ]:
keywords = ['depressie', 'isolatie', 'eenzaamheid', 'stress', 'mental',  'psychologisch', 
            'psychisch', 'mentale', 'psychische']

pscyhologie_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Description.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Title.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Category.str.contains('|'.join(keywords), case=False)
    ]

print(len(pscyhologie_articles))

pscyhologie_articles.head()

In [ ]:
keywords = ['avondklokrellen', 'avondklokboete', 'protest', 'demonstratie', 'rellen', 'demonstreren',
            'coronaprotest', 'coronarellen']

rechten_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Description.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Title.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Category.str.contains('|'.join(keywords), case=False)
    ]

print(len(rechten_articles))

rechten_articles.head()

In [ ]:
keywords = ['Tweede Kamer', 'Kamer', 'debat', 'kamervragen', 'motie', 'moties', 'kamerdebat', 'kamerlid', 'kamerleden']

politics_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Category.str.contains('|'.join(keywords), case=False)
    ]

print(len(politics_articles))

politics_articles.head()

In [ ]:
keywords = ['Oxfam', 'Verenigde Naties', 'VN', 'WHO', 'WHO-team', 'WHO-missie']

buitenland_articles = nos_notcoded_nosnieuws[
    nos_notcoded_nosnieuws.Keywords.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Category.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Title.str.contains('|'.join(keywords), case=False) |
    nos_notcoded_nosnieuws.Description.str.contains('|'.join(keywords), case=False)
    ]

print(len(buitenland_articles))

buitenland_articles.head()

In [ ]:
# merge all df with sampled articles
sampled_articles = pd.concat([long_covid_articles, 
                              misinformation_articles, 
                              medische_articles, 
                              wetenschap_articles, 
                              economie_articles, 
                              pscyhologie_articles, 
                              rechten_articles, 
                              politics_articles, 
                              buitenland_articles])

In [ ]:
# check duplicates
sampled_articles.duplicated().sum()

In [ ]:
# remove duplicates
print(len(sampled_articles))
sampled_articles = sampled_articles.drop_duplicates()
print(len(sampled_articles))

In [ ]:
# save the sampled articles
sampled_articles.to_csv('annotation/annotation_sample_docs/sampled_articles/sample_nosarticles_newset.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)